In [3]:
import math
import csv
import random
from google.colab import files

# ================= CONSTANTS & PID PARAMETERS =================
SAMPLE_INTERVAL = 0.15
Kp = 210
Ki = 390
Kd = 5.5

setVelocity          = 0.35
headwayTime          = 0.5
standstillDistance   = 0.15
pwmStepLimit         = 255
startPWM             = 90
runPWM               = 60
deadbandError        = 0.0
maxTargetVelocityDecrease = 0.03
maxIntegral          = 2.0
MAX_ACCEL            = 2.0

WMA_ALPHA_VELOCITY   = 0.25
WMA_ALPHA_DISTANCE   = 0.15

CSV_FILENAME = "telemetry_data_with_stops.csv"
CSV_HEADERS = [
    "Time(s)", "FilteredDistance(m)", "SafeDistance(m)", "DistanceError(m)",
    "TargetVelocity(m/s)", "MeasuredVelocity(m/s)", "VelocityError(m/s)",
    "PIDOutput", "PWMOutput"
]

# ================= STATE VARIABLES =================
measuredVelocity     = 0.0
filteredDistance     = 3.0
prevError            = 0.0
integral             = 0.0
lastPwmOutput        = 0
lastTargetVelocity   = setVelocity
filteredSafeDistance = standstillDistance

# ================= HELPERS =================
def wma_filter(prev, new_sample, alpha):
    if prev == 0:
        return new_sample
    return alpha * new_sample + (1 - alpha) * prev

# ================= COLAB SIMULATION LOOP =================
telemetry_log = []
simulated_time = 0.0
TARGET_ROWS = 8000

print(f"Generating {TARGET_ROWS} rows of telemetry data (Testing emergency stops)...")

for i in range(TARGET_ROWS):
    raw_d = 1.5 + 1.45 * math.sin(simulated_time / 5.0) + random.uniform(-0.05, 0.05)

    #Distance filtering
    filteredDistance = wma_filter(filteredDistance, raw_d, WMA_ALPHA_DISTANCE)

    # Simulate Robot Physics
    target_sim_vel = (lastPwmOutput / 255.0) * 0.4
    raw_v = measuredVelocity + (target_sim_vel - measuredVelocity) * 0.1 + random.uniform(-0.01, 0.01)
    measuredVelocity = wma_filter(measuredVelocity, raw_v, WMA_ALPHA_VELOCITY)
    measuredVelocity = max(0.0, measuredVelocity)

    # Safe distance & target velocity calculations
    rawSafeDistance = standstillDistance + headwayTime * measuredVelocity
    filteredSafeDistance = wma_filter(filteredSafeDistance, rawSafeDistance, 0.25)
    safeDistance  = filteredSafeDistance
    distanceError = filteredDistance - safeDistance
    v_sq = measuredVelocity ** 2 + 2 * MAX_ACCEL * distanceError
    targetVelocity = math.sqrt(max(v_sq, 0.0))
    targetVelocity = min(targetVelocity, setVelocity)

    # Safety logic: Drop target velocity to 0 if too close
    if filteredDistance <= standstillDistance:
        targetVelocity = 0.0
        integral       = 0.0

    if filteredDistance < standstillDistance * 0.8:
        targetVelocity = 0.0
        integral       = 0.0
        prevError      = 0.0

    if targetVelocity < lastTargetVelocity:
        diff = lastTargetVelocity - targetVelocity
        if diff > maxTargetVelocityDecrease:
            targetVelocity = lastTargetVelocity - maxTargetVelocityDecrease
    lastTargetVelocity = targetVelocity

    # PID Controller
    velocityError = targetVelocity - measuredVelocity
    pwmOutput     = lastPwmOutput
    pidOutput     = 0.0

    if measuredVelocity < 0.01 and lastPwmOutput >= 200 and targetVelocity > 0:
        integral = 0.0

    if abs(velocityError) >= deadbandError:
        derivative = (velocityError - prevError) / SAMPLE_INTERVAL
        prevError  = velocityError
        pidOutput  = Kp * velocityError + Ki * integral + Kd * derivative

        pwmOutput  = int(min(max(pidOutput, 0), 255))

        not_saturated = (pwmOutput < 255 or velocityError < 0) and \
                        (pwmOutput > 0  or velocityError > 0)
        if not_saturated:
            integral += velocityError * SAMPLE_INTERVAL
            integral  = max(-maxIntegral, min(integral, maxIntegral))

    if pwmOutput > lastPwmOutput + pwmStepLimit:
        pwmOutput = lastPwmOutput + pwmStepLimit
    elif pwmOutput < lastPwmOutput - pwmStepLimit:
        pwmOutput = lastPwmOutput - pwmStepLimit
    lastPwmOutput = pwmOutput

    # HARD STOP LOGIC: If the distance is near/past the threshold, kill the motors
    if filteredDistance <= standstillDistance:
        pwmOutput = 0

    # Log the data
    telemetry_log.append([
        round(simulated_time, 3),
        round(filteredDistance, 3),
        round(safeDistance, 3),
        round(distanceError, 3),
        round(targetVelocity, 3),
        round(measuredVelocity, 3),
        round(velocityError, 3),
        round(pidOutput, 1),
        pwmOutput
    ])

    # Increment time
    simulated_time += SAMPLE_INTERVAL

# ================= SAVE AND DOWNLOAD =================
print(f"Saving data to '{CSV_FILENAME}'...")
with open(CSV_FILENAME, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(CSV_HEADERS)
    writer.writerows(telemetry_log)

print("Generation complete! Triggering download...")
files.download(CSV_FILENAME)

Generating 8000 rows of telemetry data (Testing emergency stops)...
Saving data to 'telemetry_data_with_stops.csv'...
Generation complete! Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>